# பாடம் 01 - AI முகவரிகளுக்கு அறிமுகம்

**புதியவர்களுக்கான AI முகவர்கள்** பாடத்தின் முதல் பாடத்திற்கு நல்வரவு!

ஒரு **AI முகவர்** என்பது ஒரு பெரும் மொழி மாதிரியை (LLM) அதன் காரணக்கூறி என பயன்படுத்துும் ஒரு திட்டமானது மற்றும் உண்மையான உலகில் *செயல்களை* மேற்கொள்ள முடியும் — APIகளை அழைக்கும், தரவுத்தளங்களைக் கேள்வி கேட்கிறது, அல்லது குறியீட்டை இயக்குகிறது — பயனருக்காக ஒரு இலக்கை அடைய.

இந்த கணினிச்சீட்டில் நீங்கள் உங்கள் முதல் முகவரியை உருவாக்கப்போகிறீர்கள்: விடுமுறை இடங்களை பரிந்துரைக்கும் ஒரு **பயண முகவர்**. இதற்கிடையே நீங்கள் கற்பீர்கள்:

1. **Microsoft Agent Framework** பயன்படுத்தி Microsoft Foundry Agent சேவையுடன் இணைக்கவேண்டும்.
2. முகவரிக்கு ஒரு **கருவி** கொடுக்க — இது ஒரு சாதாரண Python செயல்பாடு, அதை அது அழைக்க முடியும்.
3. முகவரியை இயக்கி அதன் பதிலை ஆய்வு செய்யவும்.
4. முகவரியின் பதிலை குறியீட்டோடு-குறியீடு ஸ்ட்ரீம் செய்யவும்.


## அமைப்பு

இந்த நோட்புக் இயங்கத் தொடங்கும் முன், நீங்கள் கொண்டிருக்க வேண்டும்:

1. **மைக்ரோசாஃப்ட் ஃபௌன்ட்ரி திட்டம்** மற்றும் பரிமாறப்பட்ட அரட்டை மாதிரி (எ.கா. `gpt-4.1-mini`).
2. **ஆசூர் CLI-யில் உள்நுழைந்திருத்தல்** — உங்கள் டெர்மினலில் `az login` இயக்கவும்.
3. **தேவையான சூழல் மாறிகளை அமைக்கவும்:**
   - `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் மைக்ரோசாஃப்ட் ஃபௌன்ட்ரி திட்ட முகவரி.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — உங்கள் பரிமாறப்பட்ட மாதிரியின் பெயர்.

கீழேயுள்ள செல்லில் நீங்கள் தேவையான பய்தான் தொகுதிகள் நிறுவப்படுகின்றன.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## உங்கள் முதலும் முகவரியை உருவாக்குதல்

ஒரு முகவரிக்கு இரண்டு விஷயங்கள் வேண்டும்:

- **வழிமுறைகள்** அவனை *யார்* என்று மற்றும் *எப்படி நடக்க வேண்டும்* என்று கூறுகின்றன (ஒரு அமைப்பு அறிவுறுத்தல்).
- **கருவிகள்** — முகவரி தகவலை பெற அல்லது செயல்களை செய்ய முகவரி அழைக்கக்கூடிய `@tool` அலங்காரப்பட்ட பைதான் செயல்பாடுகள்.

கீழே நாம் பயண பரிந்துரைகள் கேட்டபோது முகவரி பயன்படுத்தும் ஒரு சாதாரண கருவியை வரையறுக்கின்றோம். இது பிரபல விடுமுறைக் கிளிக்கூட்டங்களின் பட்டியலை வழங்கும்.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ஓட்டக்காட்சி பதில்கள்

மேலும் இடையுடைமை அனுபவத்திற்காக, நீங்கள் முகவரியின் பதிலை **ஒட்டிக் காட்டலாம்**. முழு பதிலை காத்திருக்காமல், முகவர் உருவாக்கும் செய்தியின் துண்டுகளை உடனடியாக வழங்குவார். நேரலை அணுகுமுறை வெளியீட்டை உடனுக்குடன் காட்ட விரும்பும் உரையாடல் இடைமுகங்களில் இது மிகவும் பயனுள்ளதாக அமைகிறது.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி செய்வதைக் கற்றுக்கொண்டீர்கள்:

- **`FoundryChatClient` மூலம் Microsoft Foundry Agent Service-ஐ இணைக்கும் ஒரு பரப்புநிலைசெய்ய pronavida உருவாக்கவும்**.
- **இயந்திரம் உங்கள் Python செயல்பாடுகளை அழைக்க `@tool` அழகுச்சின்னத்தைப் பயன்படுத்தி ஒரு கருவியை வரையறுத்தல்**.
- **ஒரு பயனர் அழைப்புடன் இயந்திரத்தை இயக்கி அதன் பதிலை அச்சிடவும்**.
- **நேரடி வெளியீட்டிற்கான பதில்களை தொடர்ச்சியாக வழங்குதல்**.

அடுத்த பாடத்தில் நாங்கள் யந்திரக் கட்டமைப்புகளை விரிவாக ஆராய்ந்து, இயந்திரங்களுக்கு இன்னும் சக்திவாய்ந்த கருவிகள் மற்றும் பலநிலை காரணித்திறன்களை வழங்குவது எப்படி என்பதைக் கற்போம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
